# Week 6 Challenge (Lesson 159): **Beat the Baseline** with Prompt Engineering & ≤10× Larger Training Set

This notebook is a **drop-in challenge runner** for the *Product Pricer* task from Week 6.  
It builds on your existing project (`train.pkl`, `test.pkl`, and the lightweight `items`/`testing` helpers) and does three things:

1. **Baseline recap**: Re-run a strong prompting baseline (frontier model) on the test set.
2. **Fine-tuning iteration**: Prepare JSONL, upload, and fine-tune a frontier model with a **larger training subset**, **capped at 10×** the baseline size (e.g., if you used 500 examples before, you can go up to at most 5,000 examples here).
3. **Prompt-engineering upgrades**: Try **strict output formats, few-shot hints, and a small ensemble** to nudge accuracy lower than the current baseline.

> **Prerequisites**: 
> - Create a `.env` file in the same directory with your `OPENAI_API_KEY`
> - Ensure `train.pkl` and `test.pkl` exist in the current directory (generate them by running earlier week6 notebooks)
> - Have the `items.py` and `testing.py` helper modules available
>
> **Costs & access**: You need an **OpenAI API key**. Fine-tuning incurs costs. Keep your keys private and watch usage.

**Created:** 2025-09-20 10:06 UTC  
**Author:** Challenge notebook generator (LLM Engineering, Week 6 – Lesson 159)  
**Localized for:** Jupyter Lab environment

## 0) Setup (Jupyter Lab)

Run the cell below to install Python packages if needed. Make sure you have a `.env` file with your API keys.

In [1]:

# #@title Install dependencies (safe to re-run)
# !pip -q install --upgrade openai wandb python-dotenv huggingface_hub matplotlib numpy pandas tqdm nbformat



## 1) Imports


In [5]:
%matplotlib inline

In [6]:
import os, re, json, pickle, math, random, time, sys, shutil
from pathlib import Path
from typing import List, Dict, Any, Tuple, Callable
from dotenv import load_dotenv
from huggingface_hub import login

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# OpenAI client (v1+ SDK)
from openai import OpenAI

# Optional: W&B
USE_WANDB = bool(os.getenv("WANDB_API_KEY"))
if USE_WANDB:
    import wandb
    print("Weights & Biases logging enabled.")
else:
    print("Weights & Biases not enabled (set WANDB_API_KEY to use).")

Weights & Biases not enabled (set WANDB_API_KEY to use).


## 2) Environment & API keys

Set your API keys in a `.env` file in the same directory as this notebook:

```
OPENAI_API_KEY=your_openai_key_here
ANTHROPIC_API_KEY=your_anthropic_key_here  
HF_TOKEN=your_huggingface_token_here
WANDB_API_KEY=your_wandb_key_here
```

- **OpenAI**: required for fine-tuning
- **Anthropic**: optional (if using Claude models)
- **Weights & Biases**: optional for pretty training charts
- **Hugging Face**: optional for HF Hub access

In [7]:
# Load environment variables from .env file
load_dotenv(override=True)

# Set up API keys from environment
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env') 
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')

# Optional W&B setup
if not os.getenv("WANDB_API_KEY"):
    print("Optional: set WANDB_API_KEY in your .env file to log training to Weights & Biases.")

# Sanity check
assert os.getenv("OPENAI_API_KEY") and os.getenv("OPENAI_API_KEY") != 'your-key-if-not-using-env', "OPENAI_API_KEY is required. Please set it in your .env file."
print("Environment ready.")

Environment ready.


In [8]:
# Log in to HuggingFace
hf_token = os.environ['HF_TOKEN']
if hf_token and hf_token != 'your-key-if-not-using-env':
    login(hf_token, add_to_git_credential=True)
    print("Logged in to HuggingFace Hub.")
else:
    print("Optional: set HF_TOKEN in your .env file to enable HuggingFace Hub access.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to HuggingFace Hub.


In [9]:
# Initialize OpenAI client
client = OpenAI()


## 3) Load `train.pkl` and `test.pkl`

This notebook expects the same pickles used in the Week 6 day notebooks.  
If they are in Google Drive, mount and set `DATA_DIR` accordingly.


In [13]:
# Load train.pkl and test.pkl from current directory
DATA_DIR = "."
TRAIN_PKL = f"{DATA_DIR}/train.pkl"
TEST_PKL  = f"{DATA_DIR}/test.pkl"

if not Path(TRAIN_PKL).exists() or not Path(TEST_PKL).exists():
    print(f"train.pkl or test.pkl not found in current directory: {Path.cwd()}")
    print("Please ensure train.pkl and test.pkl are in the same directory as this notebook.")
    print("You can generate these files by running the earlier week6 day notebooks.")
else:
    with open(TRAIN_PKL, "rb") as f:
        train = pickle.load(f)
    with open(TEST_PKL, "rb") as f:
        test = pickle.load(f)

    print(f"Loaded train: {len(train):,} items | test: {len(test):,} items")

Loaded train: 400,000 items | test: 2,000 items



### 3.1 Helper utilities (`Item`, `Tester`) from your project

We try to import your lightweight helper classes (`items.Item`, `testing.Tester`).  
If not found, we define minimal stand-ins so the notebook still runs.


In [14]:

# Try to import your project's helpers
Item = None
Tester = None
try:
    from items import Item  # expects .test_prompt() and .price
    from testing import Tester  # expects Tester.test(func, dataset)
    print("Imported items.Item and testing.Tester from your project.")
except Exception as e:
    print("Falling back to minimal helpers:", e)
    # Minimal fallback: assume each element in train/test is a dict with keys 'text' and 'price'.
    class _FallbackItem:
        def __init__(self, text, price):
            self.text = text
            self.price = float(price)
        def test_prompt(self):
            # mimic original prompt shape
            return f"""How much does this cost to the nearest dollar?\n\n{text}\n\nPrice is $"""
    # Wrap if needed
    if len(train) > 0 and not hasattr(train[0], "test_prompt"):
        def _wrap(dataset):
            wrapped = []
            for ex in dataset:
                if isinstance(ex, dict) and "text" in ex and "price" in ex:
                    wrapped.append(_FallbackItem(ex["text"], ex["price"]))
                else:
                    raise ValueError("Fallback mode expects dicts with keys: text, price")
            return wrapped
        train = _wrap(train)
        test  = _wrap(test)
        print("Wrapped raw dicts into fallback Item-like objects.")
    # Minimal Tester
    class _FallbackTester:
        @staticmethod
        def test(fn, dataset, max_n=None):
            maes = []
            n = len(dataset) if max_n is None else min(max_n, len(dataset))
            for i in range(n):
                item = dataset[i]
                guess = fn(item)
                truth = float(item.price)
                err = abs(guess - truth)
                maes.append(err)
                color = "\x1b[92m" if err <= 50 else ("\x1b[93m" if err <= 150 else "\x1b[91m")
                print(f"{color}{i+1}: Guess: ${guess:.2f} Truth: ${truth:.2f} Error: ${err:.2f}\x1b[0m")
            print(f"\nMean Absolute Error (MAE): ${np.mean(maes):.2f} | Median: ${np.median(maes):.2f}")
    Tester = _FallbackTester

print("Helpers ready.")


Imported items.Item and testing.Tester from your project.
Helpers ready.



## 4) Shared utilities


In [15]:

def get_price(s: str) -> float:
    """Extract the first number (int/float) from a string like 'Price is $123.45'."""
    s = s.replace('$','').replace(',','')
    m = re.search(r"[-+]?\d*\.\d+|\d+", s)
    return float(m.group()) if m else 0.0



## 5) Prompt engineering variants

We provide **three** instruct styles plus a tiny **ensemble**.  
Each variant still **returns only a price**, which makes evaluation simple and reduces hallucination.


In [16]:

from dataclasses import dataclass

@dataclass
class PromptConfig:
    name: str
    system: str
    user_suffix: str
    assistant_prefix: str = "Price is $"
    few_shots: list = None  # optional few-shot exemplars: [(user_text, assistant_text), ...]
    temperature: float = 0.0
    max_tokens: int = 8

# Variant A: concise, no explanation (baseline-like)
variant_a = PromptConfig(
    name="concise_no_explanation",
    system="You estimate prices of items. Reply only with the price in USD. No explanation, no extra text.",
    user_suffix="",  # we'll use the item's test_prompt minus leading 'to the nearest dollar' artifacts
    assistant_prefix="Price is $",
    few_shots=None,
    temperature=0.0,
    max_tokens=8
)

# Variant B: strict JSON – easier to parse
variant_b = PromptConfig(
    name="strict_json",
    system='You are a pricing estimator. Return ONLY valid JSON like: {"price": 123.45}',
    user_suffix='\n\nReturn ONLY JSON like {"price": <number>} with no commentary.',
    assistant_prefix="",  # not used
    few_shots=None,
    temperature=0.0,
    max_tokens=12
)

# Variant C: few-shot priming with tiny hints
variant_c = PromptConfig(
    name="few_shot_minimal",
    system="You estimate prices. Output strictly: 'Price is $<number>' (USD). No explanation.",
    user_suffix="",
    assistant_prefix="Price is $",
    few_shots=[
        ("How much does this cost?\n\n'Wireless Mouse 2.4G, ergonomic, 1600DPI'\n\nPrice is $", "Price is $12.99"),
        ("How much does this cost?\n\n'4K HDMI cable 6ft, braided, gold-plated'\n\nPrice is $", "Price is $8.49"),
    ],
    temperature=0.0,
    max_tokens=8
)

PROMPT_VARIANTS = {
    variant_a.name: variant_a,
    variant_b.name: variant_b,
    variant_c.name: variant_c,
}
list(PROMPT_VARIANTS.keys())


['concise_no_explanation', 'strict_json', 'few_shot_minimal']

In [17]:

def build_messages(item, cfg: PromptConfig) -> list:
    """Create messages for a single item under a given prompt config."""
    # Use the project's canonical prompt, then adjust
    base_user = item.test_prompt()
    # align with lesson—remove the trailing ' to the nearest dollar' directive if present
    base_user = base_user.replace(" to the nearest dollar","").replace("\n\nPrice is $","")
    user_prompt = base_user + cfg.user_suffix

    messages = [{"role": "system", "content": cfg.system}]

    # Few-shot exemplars, if any
    if cfg.few_shots:
        for u, a in cfg.few_shots:
            messages.append({"role": "user", "content": u})
            messages.append({"role": "assistant", "content": a})

    # The actual question
    messages.append({"role": "user", "content": user_prompt})

    return messages


In [18]:

def infer_price_frontier(item, cfg: PromptConfig, model: str = "gpt-4o-mini-2024-07-18", seed: int = 42) -> float:
    """Call a frontier model with the prompt variant and return a float price."""
    msgs = build_messages(item, cfg)

    if cfg.name == "strict_json":
        # Expecting JSON back
        rsp = client.chat.completions.create(
            model=model,
            messages=msgs,
            temperature=cfg.temperature,
            max_tokens=cfg.max_tokens,
            seed=seed
        )
        txt = rsp.choices[0].message.content.strip()
        try:
            data = json.loads(txt)
            return float(data.get("price", 0))
        except Exception:
            # Fallback parse
            return get_price(txt)

    else:
        # Simple "Price is $..."
        rsp = client.chat.completions.create(
            model=model,
            messages=msgs,
            temperature=cfg.temperature,
            max_tokens=cfg.max_tokens,
            seed=seed
        )
        txt = rsp.choices[0].message.content
        if not txt.startswith(cfg.assistant_prefix):
            # be robust to small deviations
            return get_price(txt)
        return get_price(txt)


In [19]:

def ensemble_price(item, model="gpt-4o-mini-2024-07-18", variants=("concise_no_explanation","strict_json","few_shot_minimal")) -> float:
    preds = []
    for name in variants:
        cfg = PROMPT_VARIANTS[name]
        p = infer_price_frontier(item, cfg, model=model)
        preds.append(p)
    # median is robust to occasional outliers
    return float(np.median(preds))



## 6) Evaluate baselines (frontier, no fine-tune)

- **Single-variant**: choose the best-performing prompt variant.
- **Ensemble**: median across 2–3 variants (often reduces MAE modestly).


In [21]:

def evaluate_predictor(predict_fn: Callable, dataset, max_n=None, name="model"):
    maes = []
    n = len(dataset) if max_n is None else min(max_n, len(dataset))
    for i in tqdm(range(n), desc=f"Evaluating {name}"):
        item = dataset[i]
        guess = predict_fn(item)
        truth = float(item.price)
        maes.append(abs(guess - truth))
    mae = float(np.mean(maes))
    med = float(np.median(maes))
    p95 = float(np.percentile(maes, 95))
    print(f"{name} | MAE: ${mae:.2f} | Median: ${med:.2f} | 95th %ile: ${p95:.2f}")
    return np.array(maes)

# Try a quick smoke test (keep it small to avoid cost; increase after you're confident)
SMOKE = 8  # increase once happy
mae_a = evaluate_predictor(lambda it: infer_price_frontier(it, PROMPT_VARIANTS["concise_no_explanation"]), test, max_n=SMOKE, name="Variant A")
mae_b = evaluate_predictor(lambda it: infer_price_frontier(it, PROMPT_VARIANTS["strict_json"]), test, max_n=SMOKE, name="Variant B")
mae_c = evaluate_predictor(lambda it: infer_price_frontier(it, PROMPT_VARIANTS["few_shot_minimal"]), test, max_n=SMOKE, name="Variant C")
mae_ens = evaluate_predictor(lambda it: ensemble_price(it), test, max_n=SMOKE, name="Ensemble A+B+C")


Evaluating Variant A:   0%|          | 0/8 [00:00<?, ?it/s]

Variant A | MAE: $78.88 | Median: $38.75 | 95th %ile: $191.05


Evaluating Variant B:   0%|          | 0/8 [00:00<?, ?it/s]

Variant B | MAE: $53.50 | Median: $39.25 | 95th %ile: $122.83


Evaluating Variant C:   0%|          | 0/8 [00:00<?, ?it/s]

Variant C | MAE: $66.63 | Median: $39.25 | 95th %ile: $182.54


Evaluating Ensemble A+B+C:   0%|          | 0/8 [00:00<?, ?it/s]

Ensemble A+B+C | MAE: $66.50 | Median: $39.25 | 95th %ile: $182.54



## 7) Prepare a **larger** fine-tuning set (≤10× increase)

The lesson's challenge encourages using a **larger training set**, but we cap the increase at **≤10×** the earlier baseline size.

Set these knobs:
- `BASELINE_TRAIN_SIZE`: the size you used previously (e.g., 500).
- `MULTIPLE`: desired multiplier (≤10).
- We automatically compute `N_TRAIN = min(BASELINE_TRAIN_SIZE * MULTIPLE, len(train))` and keep a small validation split.


In [23]:

#@title Select training subset size (capped at 10×)
BASELINE_TRAIN_SIZE = 500  #@param {type:"number"}
MULTIPLE = 4               #@param {type:"slider", min:1, max:10, step:1}
SEED = 42                  #@param {type:"number"}

random.seed(SEED)
np.random.seed(SEED)

# Compute target
target = int(BASELINE_TRAIN_SIZE * MULTIPLE)
N_TRAIN = min(target, len(train))
assert MULTIPLE <= 10, "Cap enforced: MULTIPLE must be ≤ 10."
print(f"Requested multiple: {MULTIPLE}× of {BASELINE_TRAIN_SIZE} => target {target:,}.")
print(f"Using N_TRAIN={N_TRAIN:,} (cap ≤10× and ≤len(train)).")

# Shuffle and split
indices = np.random.permutation(len(train))
train_sel = [train[i] for i in indices[:N_TRAIN]]

# Simple validation: 10% of the chosen training subset (at least 50)
N_VAL = max(50, int(0.1 * N_TRAIN))
val_sel = train_sel[:N_VAL]
fit_sel = train_sel[N_VAL:]

len(fit_sel), len(val_sel)


Requested multiple: 4× of 500 => target 2,000.
Using N_TRAIN=2,000 (cap ≤10× and ≤len(train)).


(1800, 200)


## 8) Build JSONL for fine-tuning

We generate rows like:

```json
{"messages": [{"role":"system",...}, {"role":"user",...}, {"role":"assistant","content":"Price is $123.45"}]}
```


In [24]:

def messages_for_finetune(item, cfg: PromptConfig) -> list:
    # For supervised fine-tuning we include a known assistant answer
    sys_msg = cfg.system
    base_user = item.test_prompt().replace(" to the nearest dollar","").replace("\n\nPrice is $","")
    user_msg = base_user + cfg.user_suffix
    target = f"{cfg.assistant_prefix}{float(item.price):.2f}" if cfg.assistant_prefix else json.dumps({"price": float(item.price)})
    messages = [{"role": "system", "content": sys_msg}]

    # few-shot exemplars (optional)
    if cfg.few_shots:
        for u, a in cfg.few_shots:
            messages.append({"role": "user", "content": u})
            messages.append({"role": "assistant", "content": a})

    messages.append({"role": "user", "content": user_msg})
    messages.append({"role": "assistant", "content": target})
    return messages

def make_jsonl(items, cfg: PromptConfig) -> str:
    lines = []
    for it in items:
        msgs = messages_for_finetune(it, cfg)
        lines.append(json.dumps({"messages": msgs}, ensure_ascii=False))
    return "\n".join(lines)

def write_jsonl(items, path, cfg: PromptConfig):
    text = make_jsonl(items, cfg)
    Path(path).write_text(text, encoding="utf-8")
    print(f"Wrote {path} ({len(items):,} rows)")


In [25]:

# Choose which prompt to train on (you can try others)
FT_PROMPT_NAME = "concise_no_explanation"  #@param ["concise_no_explanation", "strict_json", "few_shot_minimal"]
ft_cfg = PROMPT_VARIANTS[FT_PROMPT_NAME]

# Filenames
OUT_DIR = Path("week6_finetune")
OUT_DIR.mkdir(exist_ok=True, parents=True)
TRAIN_JSONL = OUT_DIR / "fine_tune_train.jsonl"
VAL_JSONL   = OUT_DIR / "fine_tune_validation.jsonl"

write_jsonl(fit_sel, TRAIN_JSONL, ft_cfg)
write_jsonl(val_sel, VAL_JSONL, ft_cfg)

# Quick peek
print("\nFirst 2 lines of train JSONL:")
print("\n".join(Path(TRAIN_JSONL).read_text(encoding="utf-8").splitlines()[:2]))


Wrote week6_finetune/fine_tune_train.jsonl (1,800 rows)
Wrote week6_finetune/fine_tune_validation.jsonl (200 rows)

First 2 lines of train JSONL:
{"messages": [{"role": "system", "content": "You estimate prices of items. Reply only with the price in USD. No explanation, no extra text."}, {"role": "user", "content": "How much does this cost?\n\nSingle 9 Inch Clutch Kit Fits Ford Tractor 2N 8N 9N NAA 600 700 800 900\nOne New Aftermarket Replacement 9 Clutch Kit w/ ToolFits Ford New Holland 9 Single Clutch Tractor Models 2N, 8N, 9N, NAA, 600, 700, 800, 900Kit Contains 9 Pressure Plate HubRelease And Pilot BearingsContains Clutch Alignment ToolPlease Note For models with 3- or transmissionReplaces Part Numbers All OEM part numbers and logos are to be used for identification purposes only One New Aftermarket Replacement 9 Clutch Kit w/ Tool Fits Ford New Holland 9 Single Clutch Tractor Models 2N, 8N,"}, {"role": "assistant", "content": "Price is $116.91"}]}
{"messages": [{"role": "system", 


## 9) Upload to OpenAI & launch fine-tuning

- Model: start with `gpt-4o-mini-2024-07-18` (cheap & competitive for this task).
- Epochs: 1 (you can experiment with 2–3 if needed).

> **Tip**: Keep a record of your job IDs; enable W&B logging if you want charts.


In [19]:

from openai.types.fine_tuning import FineTuningJob

model_base = "gpt-4o-mini-2024-07-18"  # change if you want to compare to larger base

# Upload files
with open(TRAIN_JSONL, "rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")
with open(VAL_JSONL, "rb") as f:
    val_file = client.files.create(file=f, purpose="fine-tune")

print("Uploaded:", train_file.id, val_file.id)

# Optional Weights & Biases integration
integrations = []
if USE_WANDB:
    integrations = [{"type":"wandb", "wandb": {"project": "gpt-pricer-lesson159"}}]

job: FineTuningJob = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model=model_base,
    seed=SEED,
    hyperparameters={"n_epochs": 1},
    integrations=integrations,
    suffix=f"pricer_{FT_PROMPT_NAME}_{N_TRAIN}"
)
print("Created job:", job.id, "| status:", job.status)


Uploaded: file-LWpmnSCLqcYcs7em4m2yjq file-HwYQQTa8RfASmWcZXUR6kh
Created job: ftjob-i4PyBAD1XG0nmDNfwO4fVLa8 | status: validating_files


In [3]:
# Uploaded: file-LWpmnSCLqcYcs7em4m2yjq file-HwYQQTa8RfASmWcZXUR6kh
# Created job: ftjob-i4PyBAD1XG0nmDNfwO4fVLa8 | status: validating_files
job_id = "ftjob-i4PyBAD1XG0nmDNfwO4fVLa8"

In [10]:

# Poll status helper (run occasionally)
import time

def show_status(job_id, limit_events=10):
    j = client.fine_tuning.jobs.retrieve(job_id)
    print("Job:", j.id, "| status:", j.status, "| model:", j.model, "| fine_tuned:", j.fine_tuned_model)
    ev = client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=limit_events)
    for e in ev.data:
        ts = e.created_at
        print(f"- [{ts}] {e.level}: {e.message}")
    return j

# _ = show_status(job.id)
_ = show_status(job_id)

Job: ftjob-i4PyBAD1XG0nmDNfwO4fVLa8 | status: succeeded | model: gpt-4o-mini-2024-07-18 | fine_tuned: ft:gpt-4o-mini-2024-07-18:personal:pricer-concise-no-explanation-2000:CJ47OaP2
- [1758661057] info: The job has successfully completed
- [1758661048] info: Usage policy evaluations completed, model is now enabled for sampling
- [1758661047] info: Moderation checks for snapshot ft:gpt-4o-mini-2024-07-18:personal:pricer-concise-no-explanation-2000:CJ47OaP2 passed.
- [1758660099] info: Evaluating model against our usage policies
- [1758660099] info: New fine-tuned model created
- [1758660077] info: Step 1800/1800: training loss=0.48, validation loss=1.67, full validation loss=1.05
- [1758660056] info: Step 1799/1800: training loss=1.35
- [1758660056] info: Step 1798/1800: training loss=1.60
- [1758660053] info: Step 1797/1800: training loss=0.62
- [1758660053] info: Step 1796/1800: training loss=1.38



## 10) Use the fine-tuned model

After the status shows **`succeeded`** and `fine_tuned_model` is populated, use it for inference:


In [26]:

# Retrieve model name (re-run after job completes)
job_done = client.fine_tuning.jobs.retrieve(job_id)
# job_done = client.fine_tuning.jobs.retrieve(job.id)
ft_model_name = job_done.fine_tuned_model
print("Fine-tuned model:", ft_model_name)

def infer_price_finetuned(item, cfg: PromptConfig, model: str, seed: int = 42) -> float:
    msgs = build_messages(item, cfg)
    rsp = client.chat.completions.create(
        model=model,
        messages=msgs,
        temperature=cfg.temperature,
        max_tokens=cfg.max_tokens,
        seed=seed
    )
    txt = rsp.choices[0].message.content
    # robust parse
    return get_price(txt)

# Evaluate small slice first (cost-aware); then increase to your full test set.
if ft_model_name:
    _ = evaluate_predictor(lambda it: infer_price_finetuned(it, ft_cfg, ft_model_name), test, max_n=SMOKE, name="Fine-tuned (smoke)")
else:
    print("Model not ready yet. Re-run this cell after job completes.")


Fine-tuned model: ft:gpt-4o-mini-2024-07-18:personal:pricer-concise-no-explanation-2000:CJ47OaP2


Evaluating Fine-tuned (smoke):   0%|          | 0/8 [00:00<?, ?it/s]

Fine-tuned (smoke) | MAE: $58.21 | Median: $31.50 | 95th %ile: $168.89



## 11) (Optional) Small ensemble with the fine-tuned model

Median across 2–3 prompt variants often improves robustness.


In [27]:

def ensemble_price_ft(item, ft_model, variants=("concise_no_explanation","few_shot_minimal")) -> float:
    preds = []
    for name in variants:
        cfg = PROMPT_VARIANTS[name]
        preds.append(infer_price_finetuned(item, cfg, model=ft_model))
    return float(np.median(preds))

if ft_model_name:
    _ = evaluate_predictor(lambda it: ensemble_price_ft(it, ft_model_name), test, max_n=SMOKE, name="FT Ensemble (smoke)")


Evaluating FT Ensemble (smoke):   0%|          | 0/8 [00:00<?, ?it/s]

FT Ensemble (smoke) | MAE: $67.02 | Median: $53.40 | 95th %ile: $168.89



## 12) Full evaluation run (cost-aware)

Once you've verified the pipeline on a small slice, run full evaluation.  
**Remember**: Each call costs tokens. Consider batching or limiting the test size during iteration.


In [30]:

#@title Run full evaluation (set to True when ready)
RUN_FULL = False  #@param {type:"boolean"}

if RUN_FULL and ft_model_name:
    print("Running full test evaluation for baseline ensemble and fine-tuned model...")
    mae_ens_full = evaluate_predictor(lambda it: ensemble_price(it), test, name="Baseline Ensemble A+B+C")
    mae_ft_full  = evaluate_predictor(lambda it: infer_price_finetuned(it, ft_cfg, ft_model_name), test, name="Fine-tuned (single prompt)")
    mae_ft_ens   = evaluate_predictor(lambda it: ensemble_price_ft(it, ft_model_name), test, name="Fine-tuned Ensemble")
else:
    print("Set RUN_FULL=True to run full evaluations once ready (cost-aware).")


Set RUN_FULL=True to run full evaluations once ready (cost-aware).



## 13) What to try next (to **beat the baseline**)

- Try `MULTIPLE` up to **≤10×** (e.g., 10×500 = 5,000) if your `train.pkl` is large enough.
- Swap `FT_PROMPT_NAME` (train on `strict_json` or `few_shot_minimal`).
- Increase `n_epochs` to **2** or **3** if you have enough data (watch for overfitting).
- Try a larger base model after prototyping (`gpt-4o-...`) but track cost carefully.
- Keep **output strict** (JSON or `"Price is $"`) to simplify parsing.
- Keep a tiny **ensemble** (2–3 variants) for robustness.

> Challenge target from the lesson: **do better than the prior 76 USD MAE** frontier baseline (your mileage may vary).
